# StyleMatch method exploration

This notebook starts from the frozen source-heldout corpus and tests why performance differs by language, whether more independent profile evidence helps, which authorship encoders transfer, whether complementary style views improve retrieval, and whether multi-view confidence can reject unknown authors. It writes research artifacts only; it does not replace the production model or index.

Primary inference unit: independent source. Chunk results are diagnostics. Topic similarity is excluded from Style Match. Model selection uses dev; test remains locked.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
assert (REPO / 'scripts/score_artifact_utils.py').exists(), 'Pull the latest repository before running this notebook'
%cd $REPO

def run_streamed(cmd, check=True):
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    pending = b''
    while True:
        chunk = os.read(process.stdout.fileno(), 65536)
        if not chunk:
            break
        print(chunk.decode('utf-8', errors='replace').replace('\r', '\n'), end='', flush=True)
    process.wait()
    if check and process.returncode:
        raise RuntimeError(f'command failed with exit {process.returncode}: {" ".join(map(str, cmd))}')
    return process.returncode

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install pandas pyarrow sentence-transformers scikit-learn stanza

# Full core exploration fits the existing-artifact workflow. Syntax parsing and repeated fine-tuning are separable cost blocks.
RUN_SYNTAX_VIEW = True
RUN_LUAR_ENGLISH_DIAGNOSTIC = True
RUN_FINETUNE_ABLATIONS = True  # full run; set False only when reusing completed ablation artifacts
RUN_OPEN_SET = True

HF_MODELS = {
    'mstyle_pretrained': ('StyleDistance/mstyledistance', 'd66ed25e48225a503b21a65bc804caf06c886f96'),
    'challenger_pretrained': ('Blablablab/multilingual-style-representation', 'b0147bbf450424fe72c8525fcc02e2e39e3a4024'),
    # Standard SentenceTransformer conversion; English-only and diagnostic, not a multilingual replacement.
    'luar_english': ('gabrielloiseau/LUAR-MUD-sentence-transformers', '671eba4e45254d9ebdf7ba817152d0e5b1e5e1b5'),
}
EXP = Path('artifacts/method_exploration_v1')
EXP.mkdir(parents=True, exist_ok=True)

In [ ]:
# Warm the HF cache in the notebook process. Subprocess pipes are not TTYs, so
# huggingface_hub hides download progress there and a first-run download looks like a hang.
os.environ.pop('HF_HUB_ENABLE_HF_TRANSFER', None)  # deprecated since hub 0.x switch to Xet; only triggers a FutureWarning
from huggingface_hub import snapshot_download

for label, (repo_id, revision) in HF_MODELS.items():
    print(f'prefetch {label}: {repo_id}@{revision[:8]}', flush=True)
    snapshot_download(repo_id, revision=revision)
print('all pinned backbones cached', flush=True)

## 1. Reproduce aligned candidate score artifacts

All neural views use the same source-heldout rows and profile order. Hugging Face backbones are pinned to immutable commits. Local fine-tuned models are included only when their finished artifacts exist.

In [ ]:
import pandas as pd
heldout_path = REPO / 'data/all/meta/all_source_heldout_splits.parquet'
chunks_path = REPO / 'data/all/meta/all_sources_chunks.parquet'
assert heldout_path.exists() and chunks_path.exists(), 'Run the latest multilingual notebook from source fetch first'
heldout = pd.read_parquet(heldout_path)
assert {'train', 'dev', 'test'}.issubset(set(heldout['split']))
print(heldout.groupby('language')['author_or_speaker'].nunique().sort_values())

evaluation_runs = {
    'mstyle_pretrained': (*HF_MODELS['mstyle_pretrained'],),
    'challenger_pretrained': (*HF_MODELS['challenger_pretrained'],),
}
local_models = {
    'mstyle_finetuned': Path('artifacts/mstyledistance_stylematch_v1'),
    'challenger_finetuned': Path('artifacts/multilingual_author_style_v1'),
}
for name, path in local_models.items():
    if (path / 'training_config.json').exists():
        evaluation_runs[name] = (str(path), None)

embedding_specs = []
for run_name, (model_name, revision) in evaluation_runs.items():
    out_dir = EXP / f'eval_{run_name}'
    cmd = [sys.executable, 'scripts/style_embedding_recall.py', '--input', str(heldout_path), '--out-dir', str(out_dir), '--model-name', model_name, '--batch-size', '128', '--train-cap', '300', '--eval-splits', 'dev,test', '--device', 'cuda', '--skip-existing']
    if revision:
        cmd.extend(['--model-revision', revision])
    run_streamed(cmd)
    embedding_specs.extend([
        f'{run_name}_centroid={out_dir}/style_embedding_scores.npz:single_centroid_scores',
        f'{run_name}_prototype={out_dir}/style_embedding_scores.npz:source_prototype_scores',
    ])

classical_dir = EXP / 'eval_classical'
run_streamed([sys.executable, 'scripts/style_robust_baseline.py', '--input', str(heldout_path), '--out-dir', str(classical_dir), '--skip-existing'])
classical_specs = [
    f'delex_char={classical_dir}/style_robust_scores.npz:delex_char_svc',
    f'function_words={classical_dir}/style_robust_scores.npz:function_word_char_svc',
    f'rhythm_discourse={classical_dir}/style_robust_scores.npz:stylometric_logreg',
    f'compression={classical_dir}/style_robust_scores.npz:compression_distance',
]

## 2. Diagnose candidate-pool difficulty and profile evidence

The matched-N analysis compares languages at the same number of candidate authors and reports chance-adjusted MRR/Recall. The evidence curve changes the fraction of independent training sources retained per profile while holding the encoder and test sources fixed. It is not mislabeled as a fine-tuning-data experiment.

In [ ]:
difficulty_specs = [spec for spec in embedding_specs if spec.split('=', 1)[0].endswith('_centroid')]
difficulty_specs.append(f'classical_style={classical_dir}/style_robust_scores.npz:style_only_fusion')
difficulty_cmd = [sys.executable, 'scripts/evaluate_matched_difficulty.py', '--input', str(heldout_path), '--output', str(EXP / 'matched_candidate_difficulty.json'), '--candidate-sizes', '5,10,20,40,all', '--repeats', '200']
for spec in difficulty_specs:
    difficulty_cmd.extend(['--scores', spec])
run_streamed(difficulty_cmd)

for run_name in evaluation_runs:
    run_streamed([sys.executable, 'scripts/evaluate_profile_evidence_curve.py', '--input', str(heldout_path), '--eval-dir', str(EXP / f'eval_{run_name}'), '--output', str(EXP / f'profile_evidence_{run_name}.json'), '--fractions', '0.25,0.5,0.75,1.0', '--repeats', '50', '--train-cap', '300', '--split', 'test'])

## 3. Additional authorship/style views

LUAR is evaluated only in English through a standard safetensors SentenceTransformer conversion; it is not mixed into multilingual fusion. The optional syntax view contains UPOS, dependency relations/directions, and POS transitions but no lexical forms. Stanza models are downloaded only when this cell is enabled.

In [ ]:
syntax_specs = []
if RUN_SYNTAX_VIEW:
    syntax_dir = EXP / 'eval_syntax'
    run_streamed([sys.executable, 'scripts/style_syntax_baseline.py', '--input', str(heldout_path), '--out-dir', str(syntax_dir), '--device', 'cuda', '--download-missing', '--skip-existing'])
    syntax_specs = [f'syntax={syntax_dir}/style_syntax_scores.npz:syntax_scores']

if RUN_LUAR_ENGLISH_DIAGNOSTIC:
    luar_name, luar_revision = HF_MODELS['luar_english']
    luar_dir = EXP / 'eval_luar_english'
    run_streamed([sys.executable, 'scripts/style_embedding_recall.py', '--input', str(heldout_path), '--out-dir', str(luar_dir), '--model-name', luar_name, '--model-revision', luar_revision, '--language', 'en', '--batch-size', '128', '--train-cap', '300', '--eval-splits', 'dev,test', '--device', 'cuda', '--skip-existing'])
    run_streamed([sys.executable, 'scripts/evaluate_matched_difficulty.py', '--input', str(heldout_path), '--scores', f'luar_english={luar_dir}/style_embedding_scores.npz:single_centroid_scores', '--output', str(EXP / 'luar_english_matched_difficulty.json'), '--candidate-sizes', '5,10,20,40,all', '--repeats', '200'])

## 4. Optional training ablations: data scale and PCM

This block performs several new GPU trainings. The scale comparison changes source-separated pairs per eligible profile. The PCM comparison holds pair count, batching, negatives, seed, and base model fixed; PCM protects the 300 most frequent subword tokens and masks other tokens with probability 0.4, matching the released multilingual authorship code defaults. All runs remain language-aware and use matched hard negatives. Set the flag to false only when those artifacts already exist or when intentionally running the non-training diagnostics alone.

In [ ]:
ablation_specs = []
if RUN_FINETUNE_ABLATIONS:
    challenger_name, challenger_revision = HF_MODELS['challenger_pretrained']
    experiments = [
        ('pairs100', 100, 0.0),
        ('pairs250', 250, 0.0),
        ('pairs500', 500, 0.0),
        ('pairs250_pcm40', 250, 0.4),
    ]
    for label, pairs_per_profile, pcm_rate in experiments:
        model_dir = EXP / f'finetune_{label}'
        cmd = [sys.executable, 'scripts/finetune_multilingual_style.py', '--input', str(chunks_path), '--model-name', challenger_name, '--model-revision', challenger_revision, '--output-dir', str(model_dir), '--pairs-per-author', str(pairs_per_profile), '--batch-size', '16', '--epochs', '1', '--device', 'cuda', '--language-aware-batches', '--hard-negatives', '--max-seq-length', '256', '--seed', '20260710', '--skip-existing']
        if pcm_rate:
            cmd.extend(['--pcm-mask-prob', str(pcm_rate), '--pcm-num-tokens-not-to-mask', '300'])
        run_streamed(cmd)
        eval_dir = EXP / f'eval_{label}'
        run_streamed([sys.executable, 'scripts/style_embedding_recall.py', '--input', str(heldout_path), '--out-dir', str(eval_dir), '--model-name', str(model_dir), '--batch-size', '128', '--train-cap', '300', '--eval-splits', 'dev,test', '--device', 'cuda', '--skip-existing'])
        ablation_specs.append(f'{label}={eval_dir}/style_embedding_scores.npz:single_centroid_scores')

## 5. Learned multi-view Style Match

The reranker uses within-language standardized scores and candidate percentiles. It learns on dev independent sources and is evaluated once on test sources. Raw lexical character n-grams and TopicSim are deliberately excluded. Leave-one-view-out results identify whether a component adds information or only noise.

In [ ]:
fusion_specs = embedding_specs + classical_specs + syntax_specs + ablation_specs
fusion_cmd = [sys.executable, 'scripts/evaluate_multiview_fusion.py', '--input', str(heldout_path), '--output-dir', str(EXP / 'multiview_fusion'), '--bootstrap-runs', '2000', '--seed', '20260713']
for spec in fusion_specs:
    fusion_cmd.extend(['--scores', spec])
run_streamed(fusion_cmd)

## 6. Multi-view open-set verification

Entire author-language profiles are removed from the candidate index. Unknown dev profiles fit the verifier; disjoint unknown test profiles evaluate it. Confidence includes margins, score concentration, and cross-view agreement. The strictest available run uses only externally pretrained encoders; a second run includes locally fine-tuned encoders and is explicitly labeled index-open-set because those encoders saw local authors during fine-tuning.

In [ ]:
if RUN_OPEN_SET:
    strict_specs = [spec for spec in embedding_specs if ('pretrained_centroid' in spec.split('=', 1)[0])]
    index_specs = [spec for spec in embedding_specs if spec.split('=', 1)[0].endswith('_centroid')]
    eligible_languages = heldout.groupby('language')['author_or_speaker'].nunique()
    eligible_languages = sorted(eligible_languages[eligible_languages >= 10].index.astype(str))
    for language in eligible_languages:
        for label, specs in [('external_pretrained', strict_specs), ('local_index', index_specs)]:
            cmd = [sys.executable, 'scripts/evaluate_multiview_open_set.py', '--input', str(heldout_path), '--language', language, '--output-dir', str(EXP / f'open_set_{label}' / language), '--protocol-label', label, '--seed', '20260713']
            for spec in specs:
                cmd.extend(['--scores', spec])
            run_streamed(cmd)

## 7. Decision sheet

ARR-style topic regularization and EAVAE are not run here: neither has a compatible, maintained multilingual SentenceTransformer checkpoint that can be compared without building a separate training system. Their testable ideas are covered directly by matched topic/register negatives, PCM, content-free classical views, and held-out topic/domain evaluations. This is a boundary, not a zero result.

In [ ]:
fusion_report = json.loads((EXP / 'multiview_fusion/multiview_fusion_metrics.json').read_text())
print('best single:', fusion_report['best_single'])
print('locked test MRR:', fusion_report['test_metrics'])
print('paired bootstrap:', fusion_report['paired_bootstrap'])
print('decision:', fusion_report['decision'])
print('leave-one-view-out:', json.dumps(fusion_report['leave_one_view_out'], indent=2))

manifest = {
    'artifact_version': 'method_exploration_v1',
    'heldout_input': str(heldout_path),
    'hf_models': HF_MODELS,
    'syntax_run': RUN_SYNTAX_VIEW,
    'luar_english_run': RUN_LUAR_ENGLISH_DIAGNOSTIC,
    'finetune_ablations_run': RUN_FINETUNE_ABLATIONS,
    'fusion_decision': fusion_report['decision'],
    'production_changed': False,
}
(EXP / 'experiment_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('artifacts:', EXP.resolve())